# 03 · Categorical data and dimension integrity

The original exploratory notebook has been recovered unchanged in
[`archive/03_non_numeric_data_analysis_original.ipynb`](archive/03_non_numeric_data_analysis_original.ipynb).
This active notebook keeps its questions—dimension coverage, joins, types and
memory—but implements them as reproducible validations over the modelling
population, without deleting the historical exploration.


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.flight_config import DATETIME_FORMAT_PANDAS, TIME_COLUMNS

RAW = PROJECT_ROOT / "data" / "raw"
flight_path = sorted((RAW / "flights").glob("*.csv.gz"))[0]
flights = pd.read_csv(flight_path, compression="gzip")
actype = pd.read_csv(RAW / "icao" / "actype.csv")
airports_primary = pd.read_csv(RAW / "icao" / "airports.csv")
airports_secondary = pd.read_csv(RAW / "icao" / "airports2.csv", low_memory=False)
airlines = pd.read_csv(RAW / "icao" / "airlines.csv")

for column in TIME_COLUMNS:
    flights[column] = pd.to_datetime(
        flights[column], format=DATETIME_FORMAT_PANDAS, errors="coerce"
    )


## Dimension-key validation and two coverage metrics


In [2]:
def validate_dimension(fact, dimension, fact_key, dimension_key):
    dim_keys = dimension[dimension_key].dropna()
    fact_keys = fact[fact_key].dropna()
    known = set(dim_keys.unique())
    matched_rows = fact_keys.isin(known)
    unique_coverage = (
        100 * len(set(fact_keys.unique()) & known) / fact_keys.nunique()
        if fact_keys.nunique()
        else 0.0
    )
    return pd.Series({
        "dimension_rows_non_null": len(dim_keys),
        "dimension_unique_keys": dim_keys.nunique(),
        "duplicate_key_rows": dim_keys.duplicated(keep=False).sum(),
        "unique_key_coverage_pct": unique_coverage,
        "flight_weighted_coverage_pct": 100 * matched_rows.mean(),
    })


validation = pd.DataFrame({
    "aircraft_type": validate_dimension(
        flights, actype, "AC Type", "Aircraft TypeDesignator"
    ),
    "airport_departure": validate_dimension(flights, airports_primary, "ADEP", "ICAO"),
    "airport_arrival": validate_dimension(flights, airports_primary, "ADES", "ICAO"),
    "airline": validate_dimension(flights, airlines, "AC Operator", "3Ltr"),
}).T
display(validation)


,dimension_rows_non_null,dimension_unique_keys,duplicate_key_rows,unique_key_coverage_pct,flight_weighted_coverage_pct
aircraft_type,2764.0,2764.0,0.0,100.000000,100.000000
airport_departure,7508.0,7508.0,0.0,83.583155,99.219572
airport_arrival,7508.0,7508.0,0.0,83.285714,99.227815
airline,6104.0,6005.0,101.0,98.780488,99.936338


Coverage by unique keys answers “how much of the vocabulary is known?”. Weighted
coverage answers “what percentage of flights will actually match?”. Duplicate
dimension keys are a hard join failure, even when vocabulary coverage is high.


## Build a consistent airport dimension


In [3]:
primary = airports_primary.rename(columns={
    "IATA": "iata_code",
    "ICAO": "icao_code",
    "Airport name": "airport_name",
    "Country": "country_name",
    "City": "municipality",
}).assign(
    country_code=pd.NA,
    latitude=np.nan,
    longitude=np.nan,
    source="primary",
    source_priority=1,
)

secondary = airports_secondary.rename(columns={
    "name": "airport_name",
    "iso_country": "country_code",
})
coordinates = secondary["coordinates"].str.split(",", n=1, expand=True)
secondary["longitude"] = pd.to_numeric(coordinates[0], errors="coerce")
secondary["latitude"] = pd.to_numeric(coordinates[1], errors="coerce")
secondary["country_name"] = pd.NA
secondary["source"] = "secondary"
secondary["source_priority"] = 2

airport_columns = [
    "icao_code", "iata_code", "airport_name", "country_code", "country_name",
    "municipality", "latitude", "longitude", "source", "source_priority",
]
combined_airports = (
    pd.concat([primary[airport_columns], secondary[airport_columns]], ignore_index=True)
    .dropna(subset=["icao_code"])
    .sort_values(["icao_code", "source_priority"])
)

def first_non_null(series):
    values = series.dropna()
    return values.iloc[0] if len(values) else pd.NA


airports_merged = (
    combined_airports.groupby("icao_code", as_index=False)
    .agg({
        "iata_code": first_non_null,
        "airport_name": first_non_null,
        "country_code": first_non_null,
        "country_name": first_non_null,
        "municipality": first_non_null,
        "latitude": first_non_null,
        "longitude": first_non_null,
        "source": lambda values: "|".join(dict.fromkeys(values.dropna())),
    })
)

assert airports_merged["icao_code"].is_unique
display(airports_merged.head())


,icao_code,iata_code,airport_name,country_code,country_name,municipality,latitude,longitude,source
0,AGAF,AFT,Afutara Aerodrome,SB,Solomon Islands,Bila,160.948611,-9.191389,primary|secondary
1,AGAR,RNA,Ulawa Airport,SB,Solomon Islands,Arona,161.979547,-9.860544,primary|secondary
2,AGAT,ATD,Uru Harbour Airport,SB,Solomon Islands,Atoifi,161.011002,-8.87333,primary|secondary
3,AGBA,VEV,Barakoma Airport,SB,Solomon Islands,Barakoma,156.705994,-7.91278,primary|secondary
4,AGBT,BPF,Batuna Aerodrome,SB,<NA>,Batuna Mission Station,158.119306,-8.562028,secondary


`country_name` and `country_code` remain separate because combining full names
and ISO codes in one column creates inconsistent categories. Source provenance
is retained. Coordinate recovery in the Spark pipeline uses this dimension,
not hard-coded airport constants.


In [4]:
merged_validation = pd.DataFrame({
    "airport_departure": validate_dimension(flights, airports_merged, "ADEP", "icao_code"),
    "airport_arrival": validate_dimension(flights, airports_merged, "ADES", "icao_code"),
}).T
display(merged_validation)

RUN_WRITES = False
if RUN_WRITES:
    output_dir = PROJECT_ROOT / "data" / "processed" / "dimensions"
    output_dir.mkdir(parents=True, exist_ok=True)
    airports_merged.to_csv(output_dir / "airports_merged.csv", index=False)


,dimension_rows_non_null,dimension_unique_keys,duplicate_key_rows,unique_key_coverage_pct,flight_weighted_coverage_pct
airport_departure,8177.0,8177.0,0.0,85.082084,99.270782
airport_arrival,8177.0,8177.0,0.0,84.785714,99.278674


## Guarded many-to-one joins


In [5]:
def guarded_left_join(fact, dimension, fact_key, dimension_key, suffix):
    if dimension[dimension_key].isna().any():
        dimension = dimension.dropna(subset=[dimension_key])
    if not dimension[dimension_key].is_unique:
        duplicates = dimension.loc[
            dimension[dimension_key].duplicated(keep=False), dimension_key
        ].head().tolist()
        raise ValueError(f"Duplicate dimension keys for {dimension_key}: {duplicates}")

    right = dimension.rename(columns={
        column: f"{column}_{suffix}"
        for column in dimension.columns
        if column != dimension_key
    })
    before = len(fact)
    result = fact.merge(
        right,
        left_on=fact_key,
        right_on=dimension_key,
        how="left",
        validate="m:1",
    ).drop(columns=dimension_key)
    if len(result) != before:
        raise AssertionError(f"Join changed row count: {before} -> {len(result)}")
    return result


# The airline source contains duplicate/sentinel keys, so resolve them explicitly.
airlines_clean = (
    airlines.replace({"3Ltr": {"...": pd.NA}})
    .dropna(subset=["3Ltr"])
    .sort_values("3Ltr")
    .drop_duplicates("3Ltr", keep="first")
)

flights_enriched = guarded_left_join(
    flights, actype, "AC Type", "Aircraft TypeDesignator", "aircraft"
)
flights_enriched = guarded_left_join(
    flights_enriched, airports_merged, "ADEP", "icao_code", "departure"
)
flights_enriched = guarded_left_join(
    flights_enriched, airports_merged, "ADES", "icao_code", "arrival"
)
flights_enriched = guarded_left_join(
    flights_enriched, airlines_clean, "AC Operator", "3Ltr", "airline"
)
print(f"Enriched shape: {flights_enriched.shape}")


Enriched shape: (570200, 40)


In [6]:
new_columns = flights_enriched.columns.difference(flights.columns)
join_nulls = pd.DataFrame({
    "nulls": flights_enriched[new_columns].isna().sum(),
    "null_pct": (100 * flights_enriched[new_columns].isna().mean()).round(2),
}).sort_values("null_pct", ascending=False)
display(join_nulls)


,nulls,null_pct
Telephony_airline,90885,15.94
Country_airline,87132,15.28
latitude_departure,5971,1.05
longitude_departure,5971,1.05
country_code_arrival,5959,1.05
country_code_departure,5993,1.05
longitude_arrival,5937,1.04
latitude_arrival,5937,1.04
country_name_departure,4450,0.78
country_name_arrival,4403,0.77


## Explicit types and encoding strategy


In [7]:
from collections import Counter


def profile_cardinality(paths, columns, rare_threshold=1_000, chunksize=200_000):
    """Profile categorical cardinality over every period without loading all rows."""
    counters = {column: Counter() for column in columns}
    total_rows = 0

    for path in paths:
        for chunk in pd.read_csv(
            path, compression="gzip", usecols=columns, chunksize=chunksize
        ):
            total_rows += len(chunk)
            for column in columns:
                counters[column].update(chunk[column].dropna().astype(str))

    rows = []
    for column, counts in counters.items():
        frequencies = sorted(counts.values(), reverse=True)
        rows.append({
            "column": column,
            "first_period_unique": flights[column].nunique(dropna=True),
            "all_periods_unique": len(counts),
            "cardinality_ratio_pct": 100 * len(counts) / total_rows,
            "categories_below_100": sum(value < 100 for value in frequencies),
            "categories_below_1000": sum(value < rare_threshold for value in frequencies),
            "top_10_coverage_pct": 100 * sum(frequencies[:10]) / total_rows,
            "most_frequent_value": counts.most_common(1)[0][0],
            "most_frequent_pct": 100 * counts.most_common(1)[0][1] / total_rows,
        })
    return pd.DataFrame(rows).set_index("column")


cardinality_columns = [
    "ICAO Flight Type",
    "STATFOR Market Segment",
    "AC Type",
]
all_flight_paths = sorted((RAW / "flights").glob("*.csv.gz"))
cardinality_profile = profile_cardinality(all_flight_paths, cardinality_columns)
display(cardinality_profile.round(4))


,first_period_unique,all_periods_unique,cardinality_ratio_pct,categories_below_100,categories_below_1000,top_10_coverage_pct,most_frequent_value,most_frequent_pct
column,,,,,,,,
ICAO Flight Type,2,2,0.0000,0,0,100.0000,S,89.2173
STATFOR Market Segment,7,8,0.0002,0,0,100.0000,Not Classified,52.3792
AC Type,220,291,0.0071,117,163,66.0838,B738,19.7813


### Cardinality after applying the modelling-population filter


In [8]:
def profile_scheduled_cardinality(
    paths, columns, rare_threshold=1_000, chunksize=200_000
):
    """Profile categories only for scheduled commercial (`S`) flights."""
    counters = {column: Counter() for column in columns}
    total_rows = 0
    read_columns = list(dict.fromkeys([*columns, "ICAO Flight Type"]))

    for path in paths:
        for chunk in pd.read_csv(
            path, compression="gzip", usecols=read_columns, chunksize=chunksize
        ):
            chunk = chunk.loc[chunk["ICAO Flight Type"].eq("S")]
            total_rows += len(chunk)
            for column in columns:
                counters[column].update(chunk[column].dropna().astype(str))

    first_period = flights.loc[flights["ICAO Flight Type"].eq("S")]
    rows = []
    for column, counts in counters.items():
        frequencies = sorted(counts.values(), reverse=True)
        rows.append({
            "column": column,
            "first_period_unique": first_period[column].nunique(dropna=True),
            "all_periods_unique": len(counts),
            "categories_below_100": sum(value < 100 for value in frequencies),
            "categories_below_1000": sum(value < rare_threshold for value in frequencies),
            "top_10_coverage_pct": 100 * sum(frequencies[:10]) / total_rows,
        })
    return pd.DataFrame(rows).set_index("column")


scheduled_cardinality = profile_scheduled_cardinality(
    all_flight_paths, cardinality_columns
)
display(scheduled_cardinality.round(4))


,first_period_unique,all_periods_unique,categories_below_100,categories_below_1000,top_10_coverage_pct
column,,,,,
ICAO Flight Type,1,1,0,0,100.0000
STATFOR Market Segment,5,6,0,1,100.0000
AC Type,132,230,138,159,71.1487


### Encoding decision derived from the modelling population

The original 220 aircraft types were calculated on all December 2021 flights.
Across the six raw periods there are 291, but after restricting the model to
regular scheduled traffic (`ICAO Flight Type == 'S'`) there are 230. This
separation explains why raw-source and modelling cardinality differ.

- **`ICAO Flight Type` → drop after filtering.** It is constant (`S`) in the
  modelling population and therefore contains no predictive information.
- **`STATFOR Market Segment` → one-hot encoding.** It has six values in the
  selected population. `Not Classified` is retained explicitly because the later
  source periods contain no finer segment classification.
- **`AC Type` → no one-hot encoding of the raw code.** The raw code is hashed and
  rare types are grouped as `OTHER` using thresholds learned from train. After the
  aircraft join, lower-cardinality `Class_aircraft` and
  `Number+Engine Type_aircraft` are one-hot encoded for interpretability.

All stateful encoders are fitted using the training period only.


In [9]:
hash_columns = ["ADEP", "ADES", "AC Operator", "AC Type"]
hash_cardinality = profile_scheduled_cardinality(all_flight_paths, hash_columns)
unique_hash_tokens = int(hash_cardinality["all_periods_unique"].sum())

hash_candidates = pd.DataFrame({
    "num_features": [2**12, 2**13, 2**14, 2**15, 2**16],
})
hash_candidates["expected_occupied_bins"] = hash_candidates["num_features"] * (
    1 - (1 - 1 / hash_candidates["num_features"]) ** unique_hash_tokens
)
hash_candidates["expected_collision_pct"] = 100 * (
    unique_hash_tokens - hash_candidates["expected_occupied_bins"]
) / unique_hash_tokens
hash_candidates["expected_bin_load_pct"] = 100 * (
    hash_candidates["expected_occupied_bins"] / hash_candidates["num_features"]
)

SELECTED_HASH_FEATURES = 2**15
display(hash_cardinality[["first_period_unique", "all_periods_unique"]])
print(f"Scheduled-population tokens sent to hashing: {unique_hash_tokens:,}")
display(hash_candidates.set_index("num_features").round(2))


,first_period_unique,all_periods_unique
column,,
ADEP,929,1169
ADES,934,1181
AC Operator,292,412
AC Type,132,230


Scheduled-population tokens sent to hashing: 2,992


,expected_occupied_bins,expected_collision_pct,expected_bin_load_pct
num_features,,,
4096,2123.20,29.04,51.84
8192,2506.60,16.22,30.60
16384,2734.78,8.60,16.69
32768,2859.51,4.43,8.73
65536,2924.75,2.25,4.46


### Hash-vector size decision

Within scheduled commercial traffic, the four hashed columns contribute 2,992
distinct column-value tokens (`ADEP`: 1,169, `ADES`: 1,181, operator: 412 and
`AC Type`: 230). The calculation uses the uniform-hashing occupancy expectation.

`2**15 = 32,768` features remains the selected starting point: estimated token
collisions fall to approximately 4.4%, versus 8.6% at 16,384. Moving to 65,536
reduces the estimate to about 2.3% but doubles vector dimension for a smaller
absolute gain. Final selection must still compare validation quality and training
time; this calculation only supplies a defensible initial value.


In [10]:
one_hot_columns = ["STATFOR Market Segment"]
hashed_columns = ["ADEP", "ADES", "AC Operator", "AC Type"]
dropped_after_scope_filter = ["ICAO Flight Type"]
excluded_by_default = ["AC Registration"]

model_flights = flights.loc[flights["ICAO Flight Type"].eq("S")].copy()
flights_typed = model_flights.copy()
for column in one_hot_columns + hashed_columns:
    flights_typed[column] = flights_typed[column].astype("category")

memory = pd.Series({
    "before_mb": model_flights.memory_usage(deep=True).sum() / 1024**2,
    "after_category_mb": flights_typed.memory_usage(deep=True).sum() / 1024**2,
})
display(memory.to_frame("MB"))


,MB
before_mb,225.038032
after_category_mb,100.430817


Pandas `category` is a memory optimisation, not the final ML representation.
After filtering, flight type is constant and is removed. The Spark pipeline
one-hot encodes market segment and low-cardinality aircraft dimension attributes;
airport, operator and raw aircraft type are hashed. Registration is excluded by
default. Encoders are fitted only on train, preserving unseen-category handling.
